# Rwanda extraction inspection
This notebook checks the saved outputs using Python standard-library tools.
The complete archive scan, H3 boundary reconstruction, numeric conversion and profiling are in `../extract_rwanda.py`.
To rebuild first, run `uv run extract_rwanda.py` from the `fika-data` directory. That script pins its dependencies.
See `NOTES.txt` for interpretation limits. No source fields or zero values were silently imputed.


In [1]:
from pathlib import Path
import csv, json
candidates = [candidate for base in [Path.cwd(), *Path.cwd().parents]
              for candidate in [base, base / "javascript_for_planners" / "fika-data"]]
data_dir = next(p for p in candidates if (p / "extract_rwanda.py").exists())
output_dir = data_dir / "rwanda"
with (output_dir / "rwanda-hex8-attributes.csv").open() as f:
    rows = list(csv.DictReader(f))
profile = json.loads((output_dir / "quality-profile.json").read_text())
assert len(rows) == len({r["h3_index"] for r in rows}) == profile["unique_rwanda_h3_cells"]
assert all(r["country_name"] == "rwa" for r in rows)
print(f"{len(rows):,} unique Rwanda cells; {len(rows[0])} fields")
print(f"Removed tile copies: {profile['duplicate_tile_copies_removed']:,}")

24,547 unique Rwanda cells; 57 fields
Removed tile copies: 2,146


In [2]:
for category in ["all_health", "health_centers", "health_posts", "major_hospitals"]:
    field = "travel_time_no_sites_" + category
    values = [float(row[field]) for row in rows]
    assert all(v >= 0 for v in values)
    print(category, {"blank": sum(row[field] == "" for row in rows),
                     "zero": sum(v == 0 for v in values),
                     "positive": sum(v > 0 for v in values)})
print("Zeros retained; do not interpret as verified immediate access.")

all_health {'blank': 0, 'zero': 14, 'positive': 24533}
health_centers {'blank': 0, 'zero': 14, 'positive': 24533}
health_posts {'blank': 0, 'zero': 18, 'positive': 24529}
major_hospitals {'blank': 0, 'zero': 30, 'positive': 24517}
Zeros retained; do not interpret as verified immediate access.


In [3]:
for prefix in ["travel_time_", "travel_time_no_sites_", "time_delta_no_sites_"]:
    equal = sum(r[prefix + "all_health"] == r[prefix + "health_centers"] for r in rows)
    print(prefix, "all_health == health_centers:", equal)
for category, checks in profile["healthcare_scenario_checks"].items():
    assert checks["delta_mismatch_gt_one_minute"] == 0
    print(category, "maximum delta discrepancy:", checks["max_absolute_delta_discrepancy"])
print("Numeric agreement does not establish what each scenario means.")

travel_time_ all_health == health_centers: 24547
travel_time_no_sites_ all_health == health_centers: 24547
time_delta_no_sites_ all_health == health_centers: 24547
all_health maximum delta discrepancy: 1
health_centers maximum delta discrepancy: 1
health_posts maximum delta discrepancy: 1
major_hospitals maximum delta discrepancy: 1
Numeric agreement does not establish what each scenario means.


In [4]:
collection = json.loads((output_dir / "rwanda-healthcare-hex8.geojson").read_text())
assert collection["type"] == "FeatureCollection"
assert len(collection["features"]) == len(rows)
assert {f["id"] for f in collection["features"]} == {r["h3_index"] for r in rows}
assert all(f["geometry"]["coordinates"][0][0] == f["geometry"]["coordinates"][0][-1]
           for f in collection["features"])
print("GeoJSON IDs and closed rings verified; bounds:", collection["bbox"])
print("Source-cell population sum (not validated national population):",
      sum(int(r["population"]) for r in rows))

GeoJSON IDs and closed rings verified; bounds: [28.8571442, -2.8281449, 30.8853944, -1.0501613]
Source-cell population sum (not validated national population): 13439117
